# Week 5 · Probability Foundations and Bayes' Theorem

*DS 207 · Week 5*

**Assignment Notebook**

*EC Corro, built in collaboration with Claude, ChatGPT, and Gemini*

**Business scenario.** PesoWave Digital Bank is a mobile e-wallet and digital bank. A fraud-screening model flags suspicious transactions (app transfers, QR merchant payments, cash-ins, and bill payments) for review by a team of fraud analysts, and flagged customers may be sent a step-up one-time PIN (OTP) challenge. Trina Salvador, Head of Fraud and Risk Operations, relies on probability to decide how much to trust a flag, how to order the review queue, and which model settings to adopt. This assignment uses a new four-week window of PesoWave transactions ("Window B"), distinct from the window used in the lecture notes.

**Submission instructions.**

- File name: `ds207_05_probability_bayes_<lastname>.ipynb`
- Submit to your shared Google Drive folder by 11:59 PM on the day after the associated session, unless announced otherwise.

**Use of generative AI.** Generative AI is allowed for concept clarification, pseudocode, debugging, and idea generation. You must be able to explain any answer you submit without AI assistance, and you must cite any AI use.

**Points.** Section 1 (30 pts) + Section 2 (30 pts) = 60 pts.

**Dataset.** Run the cell below once before starting. It builds Window B, one row per transaction, with these fields: `channel`, `amount_band` (Low, Medium, High), `new_device` (account accessed from a device first seen within the past 24 hours), `fraud` (confirmed outcome), `flagged` (flagged by the model), and `otp_failed` (the customer failed a step-up OTP challenge; challenges are issued to flagged transactions only).

In [ ]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(207)
N_TXN = 45_000

channels = ["App Transfer", "QR Merchant Payment", "Cash-In", "Bills Payment"]
channel_share = [0.28, 0.34, 0.22, 0.16]
new_device_rate = {"App Transfer": 0.09, "QR Merchant Payment": 0.05,
                   "Cash-In": 0.07, "Bills Payment": 0.04}
base_fraud_rate = {"App Transfer": 0.014, "QR Merchant Payment": 0.008,
                   "Cash-In": 0.004, "Bills Payment": 0.006}
NEW_DEVICE_MULTIPLIER = 5
SENS, FPR = 0.90, 0.05
OTP_FAIL_FRAUD, OTP_FAIL_LEGIT = 0.75, 0.06

channel = rng.choice(channels, size=N_TXN, p=channel_share)
amount_band = rng.choice(["Low", "Medium", "High"], size=N_TXN, p=[0.65, 0.28, 0.07])
new_device = rng.random(N_TXN) < pd.Series(channel).map(new_device_rate).to_numpy()
p_fraud = (pd.Series(channel).map(base_fraud_rate).to_numpy()
           * np.where(new_device, NEW_DEVICE_MULTIPLIER, 1.0))
fraud = rng.random(N_TXN) < p_fraud
flagged = rng.random(N_TXN) < np.where(fraud, SENS, FPR)
otp_draw = rng.random(N_TXN) < np.where(fraud, OTP_FAIL_FRAUD, OTP_FAIL_LEGIT)

df = pd.DataFrame({
    "txn_id": np.arange(1, N_TXN + 1),
    "channel": channel,
    "amount_band": amount_band,
    "new_device": new_device,
    "fraud": fraud,
    "flagged": flagged,
    "otp_failed": flagged & otp_draw,
})

import matplotlib.pyplot as plt

print(df.shape)
df.head()

## Section 1: Concepts, Computation & Coding

*Three integrated problems, 10 points each (30 points in total). In each problem, part (a) is worth 3 points, part (b) 4 points, and part (c) 3 points.*

### Item 1 (10 pts): Probability rules and independence

Trina wants to know whether App Transfers carry a different fraud risk from the other channels. Let $F$ be the event that a transaction is fraud and $A$ the event that it is an App Transfer. The first 4000 transactions of Window B cross-classify as follows.

| | Fraud ($F$) | Not fraud ($F^c$) | Total |
|---|---|---|---|
| App Transfer ($A$) | 16 | 1045 | 1061 |
| Other channel ($A^c$) | 21 | 2918 | 2939 |
| Total | 37 | 3963 | 4000 |

**(a) Concept (3 pts).** Explain why $P(F \cup A)$ cannot be obtained by simply adding $P(F)$ and $P(A)$, and state what it would mean for PesoWave's fraud controls if $F$ and $A$ were independent.

**(b) By hand (4 pts).** Using the table, compute $P(F)$, $P(A)$, $P(F \cap A)$, $P(F \cup A)$, $P(F^c \cap A^c)$, and the ratio $P(F \cap A) / [P(F)\,P(A)]$. Show each step of the arithmetic.

**(c) Code (3 pts).** Reproduce your answers to (b) from the first 4000 rows of `df`. Then, using the full window, compute the same independence ratio for each of the four channels (treating each channel in turn as the event $A$), display the four ratios in a bar chart with a reference line at 1, and identify which channels deviate most from independence.

**Answer — Item 1(a)**

**Answer — Item 1(b)**

In [ ]:
# TODO

**Answer — Item 1(c)**

In [ ]:
# TODO

**Answer — Item 1(c) interpretation**

### Item 2 (10 pts): Conditional probability and the transposed question

An analyst notices that App Transfers account for a large share of flagged cases and concludes that "App Transfers are flagged far more often than other channels." Let $\text{Fl}$ be the event that a transaction is flagged. The first 4000 transactions of Window B cross-classify as follows.

| | Flagged ($\text{Fl}$) | Not flagged ($\text{Fl}^c$) | Total |
|---|---|---|---|
| App Transfer ($A$) | 70 | 991 | 1061 |
| Other channel ($A^c$) | 168 | 2771 | 2939 |
| Total | 238 | 3762 | 4000 |

**(a) Concept (3 pts).** Explain in words the difference between $P(\text{Fl} \mid A)$ and $P(A \mid \text{Fl})$, and say which of the two answers the analyst's claim about how heavily App Transfers are screened.

**(b) By hand (4 pts).** Compute $P(\text{Fl} \mid A)$, $P(A \mid \text{Fl})$, and $P(\text{Fl} \mid A^c)$. Show that $P(\text{Fl} \mid A) + P(\text{Fl} \mid A^c)$ does not equal 1, and verify the multiplication rule by computing $P(\text{Fl} \cap A)$ two ways.

**(c) Code (3 pts).** Reproduce (b) from the first 4000 rows of `df`. Then, using the full window, compute $P(\text{Fl} \mid \text{channel})$ for all four channels alongside the overall flag rate, plot them as a bar chart with a reference line at the overall rate, and compare the full-window $P(A \mid \text{Fl})$ with $P(A)$.

**Answer — Item 2(a)**

**Answer — Item 2(b)**

In [ ]:
# TODO

**Answer — Item 2(c)**

In [ ]:
# TODO

**Answer — Item 2(c) interpretation**

### Item 3 (10 pts): Bayes' theorem and the base rate

PesoWave's validation report for the current model states a sensitivity $P(\text{Fl} \mid F) = 0.90$ and a false-positive rate $P(\text{Fl} \mid F^c) = 0.05$. Across the platform the fraud rate is approximately $P(F) = 0.011$.

**(a) Concept (3 pts).** A colleague says, "The model catches 90% of fraud, so 90% of flagged transactions must be fraud." Explain why this reasoning fails. In your explanation, identify which of the three numbers above plays the role of the prior and which are likelihoods.

**(b) By hand (4 pts).** Compute $P(F \mid \text{Fl})$ using Bayes' theorem, showing the numerator and the law-of-total-probability denominator. Then express the same result in natural frequencies for 100,000 transactions and state how many flagged cases an analyst opens, on average, per genuine fraud found.

**(c) Code (3 pts).** Write a function that returns the posterior from a prior, a sensitivity, and a false-positive rate, and confirm your hand result. Compare it with the empirical sensitivity, false-positive rate, and share of flagged cases that are fraud in `df`. Plot the posterior against the prior for priors between 0.1% and 30% (log-scale horizontal axis), mark the prior at which the posterior first reaches 0.50, and mark PesoWave's observed prior.

**Answer — Item 3(a)**

**Answer — Item 3(b)**

In [ ]:
# TODO

**Answer — Item 3(c)**

In [ ]:
# TODO

**Answer — Item 3(c) interpretation**

## Section 2: Business Context & Reasoning

*Two stakeholder problems, 15 points each (30 points in total). In each problem, part (a) is worth 3 points, part (b) 6 points, and part (c) 6 points. Parts (a) and (b) are assessed for technical soundness; part (c) is assessed for business reasoning. Work that is technically correct but yields a recommendation that ignores the operating context, and work with a sensible recommendation built on incorrect probabilities, both fall short of full marks.*

### Item 4 (15 pts): Choosing a model operating point under limited review capacity

Trina must choose between two operating points for the fraud model next month. Treat Window B as a representative month of 45,000 transactions, with a fraud rate equal to the rate observed in `df`.

| Option | Sensitivity | False-positive rate |
|---|---|---|
| A (current threshold) | 0.90 | 0.05 |
| B (stricter threshold) | 0.80 | 0.02 |

Operating facts: each manual review of a flagged case costs ₱95; each fraud that is not stopped costs ₱18,500 on average; analysts can open at most 1,800 flagged cases per month. Flagged cases are opened in random order until capacity is reached. A fraud in an opened case is stopped. Flagged cases beyond capacity are released, and any fraud among them is lost, as is every fraud the model fails to flag. Total cost is the review cost of the opened cases plus the loss from frauds that are not stopped.

**(a) Assumption (3 pts).** State one assumption, beyond those given above, on which your comparison depends, and explain briefly why it is reasonable for PesoWave.

**(b) Technique and interpretation (6 pts).** For each option, compute the expected number of flagged cases, the expected number of genuine frauds among them, the posterior $P(F \mid \text{Fl})$, the number of cases opened, the number of frauds stopped and not stopped, and the total cost. Interpret in two or three sentences why the two options differ in the share of flagged cases that are fraud.

**(c) Recommendation (6 pts).** Write a recommendation to Trina (150 words or fewer) that states which option you would adopt, addresses the capacity constraint, and names one risk or follow-up action.

**Answer — Item 4(a)**

**Answer — Item 4(b) computation**

In [ ]:
# TODO

**Answer — Item 4(b) interpretation and 4(c) recommendation**

### Item 5 (15 pts): Routing flagged cases using a sequential update

PesoWave routes each flagged transaction using the probability that it is fraud after the model flag and the step-up OTP result: a posterior of 0.60 or higher is placed on an automatic hold; a posterior of 0.05 or lower is released automatically; anything in between joins the analyst queue. The model has sensitivity 0.90 and false-positive rate 0.05. The OTP challenge is failed by 75% of fraudulent attempts and 6% of legitimate ones. Use the observed fraud rate of each channel-by-device segment in `df` as the prior for that segment.

Three flagged transactions arrive:

- **Case 1:** App Transfer, new device, OTP failed.
- **Case 2:** QR Merchant Payment, established device, OTP passed.
- **Case 3:** Cash-In, established device, OTP failed.

**(a) Assumption (3 pts).** The sequential update treats the model flag and the OTP result as conditionally independent given the fraud status. State this assumption in your own words and explain one realistic way it could fail at PesoWave.

**(b) Technique and interpretation (6 pts).** Compute the segment prior, the posterior after the flag, and the posterior after the OTP result for each case. Compare each final posterior with the observed share of fraud among flagged cases in `df` with the same segment and OTP outcome, and comment on any gap, including how many cases lie behind each observed figure.

**(c) Recommendation (6 pts).** State the routing (automatic hold, analyst queue, or automatic release) for each case and justify it. Then recommend one safeguard for customers who are legitimately held, and one metric Trina should monitor to confirm the routing rule is working.

**Answer — Item 5(a)**

**Answer — Item 5(b) computation**

In [ ]:
# TODO

**Answer — Item 5(b) interpretation and 5(c) recommendation**

## Citation Reminder

All external sources used in this assignment (textbooks, papers, articles, code snippets, and any use of generative AI tools) must be cited in accordance with the course citation policy. Add each additional source you use to the References section below.

## References

Blitzstein, J. K., & Hwang, J. (2019). *Introduction to Probability* (2nd ed.). CRC Press.

Gigerenzer, G., & Hoffrage, U. (1995). How to improve Bayesian reasoning without instruction: Frequency formats. *Psychological Review, 102*(4), 684-704.

Ross, S. M. (2019). *A First Course in Probability* (10th ed.). Pearson.

Wasserman, L. (2004). *All of Statistics: A Concise Course in Statistical Inference*. Springer.